In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    trim,
    lower,
    upper,
    regexp_replace,
    when,
    lit,
    to_date,
    year,
    monotonically_increasing_id
)
from pyspark.sql.types import DoubleType

spark = SparkSession.builder \
    .appName("IndianStartupFunding_Silver") \
    .getOrCreate()

storage_account = "startupfundingstorage"

bronze_path = (
    f"abfss://bronze@{storage_account}.dfs.core.windows.net/"
    "startup_funding"
)

silver_path = (
    f"abfss://silver@{storage_account}.dfs.core.windows.net/"
    "startup_funding"
)

print("Reading Bronze Delta data...")

bronze_df = spark.read \
    .format("delta") \
    .load(bronze_path)

print("Bronze row count:", bronze_df.count())

print("Bronze columns:")
print(bronze_df.columns)

display(bronze_df)


# ---------------------------------------------------------
# STEP 1: STANDARDIZE COLUMN NAMES
# ---------------------------------------------------------

for column in bronze_df.columns:
    new_column = (
        column.strip()
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
        .replace("/", "_")
        .replace("(", "")
        .replace(")", "")
        .replace(".", "")
    )

    bronze_df = bronze_df.withColumnRenamed(column, new_column)

print("Standardized columns:")
print(bronze_df.columns)


# ---------------------------------------------------------
# STEP 2: IDENTIFY DATASET COLUMNS
# ---------------------------------------------------------

date_col = next(
    (c for c in bronze_df.columns if c in ["date", "date_dd_mm_yyyy"]),
    None
)

startup_col = next(
    (c for c in bronze_df.columns if "startup" in c),
    None
)

industry_col = next(
    (
        c for c in bronze_df.columns
        if "industry" in c or "vertical" in c
    ),
    None
)

subvertical_col = next(
    (
        c for c in bronze_df.columns
        if "sub" in c and ("vertical" in c or "sector" in c)
    ),
    None
)

city_col = next(
    (c for c in bronze_df.columns if c == "city" or "city" in c),
    None
)

investor_col = next(
    (
        c for c in bronze_df.columns
        if "investor" in c
    ),
    None
)

investment_type_col = next(
    (
        c for c in bronze_df.columns
        if "investment" in c and "type" in c
    ),
    None
)

amount_col = next(
    (
        c for c in bronze_df.columns
        if "amount" in c
    ),
    None
)

print("Detected columns:")
print("Date:", date_col)
print("Startup:", startup_col)
print("Industry:", industry_col)
print("Sub-Vertical:", subvertical_col)
print("City:", city_col)
print("Investor:", investor_col)
print("Investment Type:", investment_type_col)
print("Amount:", amount_col)


# ---------------------------------------------------------
# STEP 3: RENAME COLUMNS TO STANDARD NAMES
# ---------------------------------------------------------

rename_mapping = {}

if date_col:
    rename_mapping[date_col] = "funding_date"

if startup_col:
    rename_mapping[startup_col] = "startup_name"

if industry_col:
    rename_mapping[industry_col] = "industry_vertical"

if subvertical_col:
    rename_mapping[subvertical_col] = "sub_vertical"

if city_col:
    rename_mapping[city_col] = "city"

if investor_col:
    rename_mapping[investor_col] = "investor_names"

if investment_type_col:
    rename_mapping[investment_type_col] = "investment_type"

if amount_col:
    rename_mapping[amount_col] = "amount_usd"

for old_name, new_name in rename_mapping.items():
    if old_name != new_name:
        bronze_df = bronze_df.withColumnRenamed(
            old_name,
            new_name
        )

print("Final standardized columns:")
print(bronze_df.columns)


# ---------------------------------------------------------
# STEP 4: TRIM STRING COLUMNS
# ---------------------------------------------------------

string_columns = [
    field.name
    for field in bronze_df.schema.fields
    if field.dataType.simpleString() == "string"
]

for column_name in string_columns:
    bronze_df = bronze_df.withColumn(
        column_name,
        trim(col(column_name))
    )


# ---------------------------------------------------------
# STEP 5: HANDLE EMPTY STRINGS
# ---------------------------------------------------------

for column_name in string_columns:
    bronze_df = bronze_df.withColumn(
        column_name,
        when(
            col(column_name) == "",
            None
        ).otherwise(col(column_name))
    )


# ---------------------------------------------------------
# STEP 6: DATE PARSING
# ---------------------------------------------------------

if "funding_date" in bronze_df.columns:

    bronze_df = bronze_df.withColumn(
        "funding_date",
        when(
            col("funding_date").rlike(r"^\d{1,2}/\d{1,2}/\d{4}$"),
            to_date(col("funding_date"), "d/M/yyyy")
        )
        .when(
            col("funding_date").rlike(r"^\d{1,2}-\d{1,2}-\d{4}$"),
            to_date(col("funding_date"), "d-M-yyyy")
        )
        .when(
            col("funding_date").rlike(r"^\d{4}-\d{1,2}-\d{1,2}$"),
            to_date(col("funding_date"), "yyyy-M-d")
        )
        .otherwise(None)
    )


# ---------------------------------------------------------
# STEP 7: ADD FUNDING YEAR
# ---------------------------------------------------------

if "funding_date" in bronze_df.columns:

    bronze_df = bronze_df.withColumn(
        "funding_year",
        year(col("funding_date"))
    )


# ---------------------------------------------------------
# STEP 8: NORMALIZE CITY NAMES
# ---------------------------------------------------------

if "city" in bronze_df.columns:

    bronze_df = bronze_df.withColumn(
        "city",
        trim(col("city"))
    )

    bronze_df = bronze_df.withColumn(
        "city",
        when(
            lower(col("city")).isin(
                "bangalore",
                "bengaluru"
            ),
            "Bengaluru"
        )
        .when(
            lower(col("city")).isin(
                "new delhi",
                "delhi"
            ),
            "Delhi"
        )
        .when(
            lower(col("city")).isin(
                "mumbai",
                "bombay"
            ),
            "Mumbai"
        )
        .when(
            lower(col("city")).isin(
                "gurgaon",
                "gurugram"
            ),
            "Gurugram"
        )
        .when(
            lower(col("city")).isin(
                "noida"
            ),
            "Noida"
        )
        .otherwise(col("city"))
    )


# ---------------------------------------------------------
# STEP 9: STANDARDIZE INDUSTRY VERTICAL
# ---------------------------------------------------------

if "industry_vertical" in bronze_df.columns:

    bronze_df = bronze_df.withColumn(
        "industry_vertical",
        trim(col("industry_vertical"))
    )

    bronze_df = bronze_df.withColumn(
        "industry_vertical",
        when(
            col("industry_vertical").isNull()
            | (col("industry_vertical") == ""),
            "Unknown"
        )
        .otherwise(col("industry_vertical"))
    )


# ---------------------------------------------------------
# STEP 10: STANDARDIZE SUB-VERTICAL
# ---------------------------------------------------------

if "sub_vertical" in bronze_df.columns:

    bronze_df = bronze_df.withColumn(
        "sub_vertical",
        when(
            col("sub_vertical").isNull()
            | (col("sub_vertical") == ""),
            "Unknown"
        )
        .otherwise(trim(col("sub_vertical")))
    )


# ---------------------------------------------------------
# STEP 11: STANDARDIZE INVESTOR NAMES
# ---------------------------------------------------------

if "investor_names" in bronze_df.columns:

    bronze_df = bronze_df.withColumn(
        "investor_names",
        when(
            col("investor_names").isNull()
            | (col("investor_names") == ""),
            "Unknown"
        )
        .otherwise(trim(col("investor_names")))
    )


# ---------------------------------------------------------
# STEP 12: STANDARDIZE INVESTMENT TYPE
# ---------------------------------------------------------

if "investment_type" in bronze_df.columns:

    bronze_df = bronze_df.withColumn(
        "investment_type",
        when(
            col("investment_type").isNull()
            | (col("investment_type") == ""),
            "Unknown"
        )
        .otherwise(trim(col("investment_type")))
    )


# ---------------------------------------------------------
# STEP 13: CLEAN AND STANDARDIZE AMOUNT
# ---------------------------------------------------------

if "amount_usd" in bronze_df.columns:

    bronze_df = bronze_df.withColumn(
        "amount_usd",
        regexp_replace(
            col("amount_usd"),
            r"[\$,]",
            ""
        )
    )

    bronze_df = bronze_df.withColumn(
        "amount_usd",
        regexp_replace(
            col("amount_usd"),
            r"\s+",
            ""
        )
    )

    bronze_df = bronze_df.withColumn(
        "amount_usd",
        when(
            lower(col("amount_usd")).isin(
                "null",
                "na",
                "n/a",
                "nan",
                ""
            ),
            None
        ).otherwise(col("amount_usd"))
    )

    bronze_df = bronze_df.withColumn(
        "amount_usd",
        col("amount_usd").cast(DoubleType())
    )


# ---------------------------------------------------------
# STEP 14: REMOVE DUPLICATES
# ---------------------------------------------------------

before_dedup = bronze_df.count()

bronze_df = bronze_df.dropDuplicates()

after_dedup = bronze_df.count()

print("Rows before deduplication:", before_dedup)
print("Rows after deduplication:", after_dedup)
print("Duplicates removed:", before_dedup - after_dedup)


# ---------------------------------------------------------
# STEP 15: FINAL NULL HANDLING
# ---------------------------------------------------------

if "startup_name" in bronze_df.columns:

    bronze_df = bronze_df.filter(
        col("startup_name").isNotNull()
        & (trim(col("startup_name")) != "")
    )

if "funding_date" in bronze_df.columns:

    bronze_df = bronze_df.filter(
        col("funding_date").isNotNull()
    )


# ---------------------------------------------------------
# STEP 16: FINAL COLUMN ORDER
# ---------------------------------------------------------

preferred_columns = [
    "funding_date",
    "funding_year",
    "startup_name",
    "industry_vertical",
    "sub_vertical",
    "city",
    "investor_names",
    "investment_type",
    "amount_usd"
]

existing_preferred_columns = [
    c for c in preferred_columns
    if c in bronze_df.columns
]

remaining_columns = [
    c for c in bronze_df.columns
    if c not in existing_preferred_columns
]

bronze_df = bronze_df.select(
    existing_preferred_columns + remaining_columns
)


# ---------------------------------------------------------
# STEP 17: DISPLAY SILVER DATA
# ---------------------------------------------------------

print("========== SILVER DATA ==========")

print("Silver Row Count:")
print(bronze_df.count())

print("Silver Schema:")
bronze_df.printSchema()

display(bronze_df)


# ---------------------------------------------------------
# STEP 18: WRITE SILVER DELTA
# ---------------------------------------------------------

bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_path)

print("Silver Delta data written successfully.")


# ---------------------------------------------------------
# STEP 19: READ SILVER DATA FOR VALIDATION
# ---------------------------------------------------------

silver_df = spark.read \
    .format("delta") \
    .load(silver_path)

print("========== SILVER VALIDATION ==========")

print("Silver Row Count:", silver_df.count())

print("Silver Column Count:", len(silver_df.columns))

print("Silver Columns:")
print(silver_df.columns)

print("Silver Schema:")
silver_df.printSchema()

display(silver_df)


# ---------------------------------------------------------
# STEP 20: NULL VALIDATION
# ---------------------------------------------------------

from pyspark.sql.functions import sum as spark_sum

print("========== NULL VALIDATION ==========")

null_counts = silver_df.select([
    spark_sum(
        when(col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in silver_df.columns
])

display(null_counts)


# ---------------------------------------------------------
# STEP 21: DUPLICATE VALIDATION
# ---------------------------------------------------------

total_rows = silver_df.count()
distinct_rows = silver_df.dropDuplicates().count()

print("========== DUPLICATE VALIDATION ==========")

print("Total Rows:", total_rows)
print("Distinct Rows:", distinct_rows)
print("Duplicate Rows:", total_rows - distinct_rows)


# ---------------------------------------------------------
# STEP 22: FINAL SUCCESS MESSAGE
# ---------------------------------------------------------

print("==========================================")
print("SILVER LAYER COMPLETED SUCCESSFULLY")
print("==========================================")
print("Bronze Rows:", before_dedup)
print("Silver Rows:", silver_df.count())
print("Silver Columns:", len(silver_df.columns))
print("Silver Delta Path:", silver_path)
print("==========================================")

Reading Bronze Delta data...
Bronze row count: 1100
Bronze columns:
['Startup', 'Industry', 'SubVertical', 'City', 'Investors', 'InvestmentType', 'InvestmentAmount_USD', 'Date']


Startup,Industry,SubVertical,City,Investors,InvestmentType,InvestmentAmount_USD,Date
Housejoy,EdTech,K12,Mumbai,Lightspeed India,Seed,199000,19-04-2023
Groww,Media,Streaming,Bengaluru,IFC,Seed,1668000,28-01-2025
Groww,Mobility,Ride Sharing,Hyderabad,"Nexus Venture Partners, Peak XV",Series B,38052000,14-03-2021
FarmBox,Consumer Electronics,Wearables,Gurugram,"Kalaari Capital, Y Combinator",Seed,455000,11-09-2023
Udaan,RealEstate,Rental Tech,Mumbai,Bessemer Venture Partners,Seed,89000,31-01-2024
AeroStack,EdTech,Coding Bootcamp,Delhi,"Matrix Partners India, Peak XV",Pre-Series A,143000,24-07-2021
Ola,Retail,Fashion,Mumbai,Tiger Global Management,Seed,62000,12-01-2023
FinSpace,Consumer Electronics,Wearables,Gurugram,Blume Ventures,Growth,491721000,22-01-2025
Rapido,FoodTech,Food Delivery,Kolkata,"A91 Partners, Kedaara Capital, Tiger Global Management",Seed,459000,27-11-2023
HyperLoop,EdTech,Test Prep,Ahmedabad,"Mirae Asset, SoftBank Vision Fund, Tiger Global",Series B,35610000,04-05-2020


Standardized columns:
['startup', 'industry', 'subvertical', 'city', 'investors', 'investmenttype', 'investmentamount_usd', 'date']
Detected columns:
Date: date
Startup: startup
Industry: industry
Sub-Vertical: subvertical
City: city
Investor: investors
Investment Type: investmenttype
Amount: investmentamount_usd
Final standardized columns:
['startup_name', 'industry_vertical', 'sub_vertical', 'city', 'investor_names', 'investment_type', 'amount_usd', 'funding_date']
Rows before deduplication: 1100
Rows after deduplication: 1100
Duplicates removed: 0
========== SILVER DATA ==========
Silver Row Count:
1100
Silver Schema:
root
 |-- funding_date: date (nullable = true)
 |-- funding_year: integer (nullable = true)
 |-- startup_name: string (nullable = true)
 |-- industry_vertical: string (nullable = true)
 |-- sub_vertical: string (nullable = true)
 |-- city: string (nullable = true)
 |-- investor_names: string (nullable = true)
 |-- investment_type: string (nullable = true)
 |-- amount_u

funding_date,funding_year,startup_name,industry_vertical,sub_vertical,city,investor_names,investment_type,amount_usd
2020-02-26,2020,QuantumSolutions,Mobility,EV,Noida,IFC,Seed,104000.0
2021-01-24,2021,HyperLabs,Mobility,Ride Sharing,Chennai,Tiger Global,Angel,29000.0
2020-02-22,2020,Porter,Retail,E-Retail,Hyderabad,A91 Partners,Seed,1788000.0
2021-01-25,2021,AgriHive,EdTech,Coding Bootcamp,Noida,"Falcon Edge, IFC",Seed,157000.0
2024-01-16,2024,FreshBox,FoodTech,Food Delivery,Chennai,Tiger Global Management,Growth,1.54344E8
2023-10-17,2023,Dream11,Media,Content,Bengaluru,"Elevation Capital, Info Edge, Matrix Partners India",Pre-Series A,175000.0
2021-01-28,2021,AgriFit,Enterprise,Automation,Hyderabad,Elevation Capital,Series A,6512000.0
2022-05-08,2022,FoodSolutions,EdTech,Test Prep,Bengaluru,"IFC, Ventures India",Series C,1.14859E8
2024-02-09,2024,BlueSpace,Enterprise,Security,Kolkata,Blume Ventures,Series A,2359000.0
2020-01-26,2020,Bounce,AgriTech,Marketplace,Pune,Kalaari Capital,Growth,5.5675E8


Silver Delta data written successfully.
========== SILVER VALIDATION ==========
Silver Row Count: 1100
Silver Column Count: 9
Silver Columns:
['funding_date', 'funding_year', 'startup_name', 'industry_vertical', 'sub_vertical', 'city', 'investor_names', 'investment_type', 'amount_usd']
Silver Schema:
root
 |-- funding_date: date (nullable = true)
 |-- funding_year: integer (nullable = true)
 |-- startup_name: string (nullable = true)
 |-- industry_vertical: string (nullable = true)
 |-- sub_vertical: string (nullable = true)
 |-- city: string (nullable = true)
 |-- investor_names: string (nullable = true)
 |-- investment_type: string (nullable = true)
 |-- amount_usd: double (nullable = true)



funding_date,funding_year,startup_name,industry_vertical,sub_vertical,city,investor_names,investment_type,amount_usd
2020-02-26,2020,QuantumSolutions,Mobility,EV,Noida,IFC,Seed,104000.0
2021-01-24,2021,HyperLabs,Mobility,Ride Sharing,Chennai,Tiger Global,Angel,29000.0
2020-02-22,2020,Porter,Retail,E-Retail,Hyderabad,A91 Partners,Seed,1788000.0
2021-01-25,2021,AgriHive,EdTech,Coding Bootcamp,Noida,"Falcon Edge, IFC",Seed,157000.0
2024-01-16,2024,FreshBox,FoodTech,Food Delivery,Chennai,Tiger Global Management,Growth,1.54344E8
2023-10-17,2023,Dream11,Media,Content,Bengaluru,"Elevation Capital, Info Edge, Matrix Partners India",Pre-Series A,175000.0
2021-01-28,2021,AgriFit,Enterprise,Automation,Hyderabad,Elevation Capital,Series A,6512000.0
2022-05-08,2022,FoodSolutions,EdTech,Test Prep,Bengaluru,"IFC, Ventures India",Series C,1.14859E8
2024-02-09,2024,BlueSpace,Enterprise,Security,Kolkata,Blume Ventures,Series A,2359000.0
2020-01-26,2020,Bounce,AgriTech,Marketplace,Pune,Kalaari Capital,Growth,5.5675E8


========== NULL VALIDATION ==========


funding_date,funding_year,startup_name,industry_vertical,sub_vertical,city,investor_names,investment_type,amount_usd
0,0,0,0,0,0,0,0,0


========== DUPLICATE VALIDATION ==========
Total Rows: 1100
Distinct Rows: 1100
Duplicate Rows: 0
SILVER LAYER COMPLETED SUCCESSFULLY
Bronze Rows: 1100
Silver Rows: 1100
Silver Columns: 9
Silver Delta Path: abfss://silver@startupfundingstorage.dfs.core.windows.net/startup_funding
